# Gravity Model 

### What is gravity model?

The gravity model estimates how strongly a location “attracts” people based on nearby population and distance.  
- **Nearby population:** More people leads to higher influence.  
- **Distance decay:** Influence decreases with distance (controlled by beta).   

Gravity Score = Σ (Population_i / Distance_i^β)

- **Population\_i:** Population at nearby location \(i\)  
- **Distance\_i:** Distance from candidate point to location \(i\)  
- **Beta (β = 1.5):** Distance decay exponent, controls how quickly influence decreases with distance  

Used here to score candidate locations for new Tube stations — **higher gravity score = more potential users within 2.5 km**.

## Import Libraries

In [14]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
from sklearn.neighbors import BallTree

## Load Data

In [12]:
# Population and coordinates
population = pd.read_csv("Dataset/Population.csv", sep=";", usecols=[0,2])
population = population.rename(columns={'All Ages': 'Population', 'OA11CD':'Area'})

coordinates = pd.read_csv("Dataset/Coordinate_filter.csv", sep=",")

# Existing stations
stations = pd.read_csv("Dataset/combined_stations.csv")

# Merge population & coordinates
data = pd.merge(population, coordinates, on='Area')

## Convert to GeoDataFrames

In [4]:
# Population points
gdf = gpd.GeoDataFrame(
    data,
    geometry=[Point(xy) for xy in zip(data['LONG'], data['LAT'])],
    crs="EPSG:4326"
)

# Stations
stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=[Point(xy) for xy in zip(stations['lon'], stations['lat'])],
    crs="EPSG:4326"
)

# Convert to metric CRS
gdf = gdf.to_crs(epsg=27700)
stations_gdf = stations_gdf.to_crs(epsg=27700)

## Create Candidate Grid

In [5]:
# Grid creation
xmin, ymin, xmax, ymax = gdf.total_bounds
step = 1000  # 1 km grid
grid_points = [Point(x, y) for x in np.arange(xmin, xmax, step)
                        for y in np.arange(ymin, ymax, step)]
grid = gpd.GeoDataFrame(geometry=grid_points, crs=gdf.crs)

## Filter Grid Near Existing Stations
**Station exclusion radius (800 m):** Prevents new stations from being too close to existing stations.  

In [6]:
# Build BallTree for stations
station_coords = np.array([[pt.x, pt.y] for pt in stations_gdf.geometry])
station_tree = BallTree(station_coords, metric='euclidean')

# Filter out grid points within 800 m
grid_coords = np.array([[pt.x, pt.y] for pt in grid.geometry])
idxs = station_tree.query_radius(grid_coords, r=800)
mask = np.array([len(i) == 0 for i in idxs])
grid = grid[mask].reset_index(drop=True)

## Compute Gravity Scores

### Gravity Model Parameters
- **Radius (2500 m):** Defines how far we consider populations to influence a candidate station.  
- **Beta (1.5):** Distance decay factor; larger beta emphasizes local population density more strongly than distant populations.

In [7]:
# Prepare BallTree for population
pop_coords = np.array([[pt.x, pt.y] for pt in gdf.geometry])
pop_values = gdf['Population'].values
tree = BallTree(pop_coords, metric='euclidean')

grid_coords = np.array([[pt.x, pt.y] for pt in grid.geometry])

radius = 2500
beta = 1.5

idxs_list, dists_list = tree.query_radius(grid_coords, r=radius, return_distance=True)

gravity_scores = []
for idxs, d in zip(idxs_list, dists_list):
    idxs = np.array(idxs, dtype=int)
    d = np.array(d)
    if len(idxs) == 0:
        gravity_scores.append(0)
        continue
    d = np.where(d == 0, 1e-6, d)
    score = (pop_values[idxs] / (d ** beta)).sum()
    gravity_scores.append(score)

grid['gravity_score'] = np.array(gravity_scores)
grid['gravity_log'] = np.log1p(grid['gravity_score'])

Convert to Lat/Lon Coordinate

In [15]:
grid_latlon = grid.to_crs(epsg=4326)
grid_latlon['lat'] = grid_latlon.geometry.y
grid_latlon['lon'] = grid_latlon.geometry.x
grid_latlon = grid_latlon.sort_values('gravity_log', ascending=False)
print(grid_latlon)

                       geometry  gravity_score  gravity_log        lat  \
254   POINT (-0.41251 51.51452)      99.490279     4.610061  51.514517   
863   POINT (-0.14541 51.34873)      29.916822     3.431300  51.348731   
465   POINT (-0.32806 51.45939)      16.619460     2.869004  51.459390   
1250    POINT (0.04475 51.4265)      16.334886     2.852721  51.426502   
1512    POINT (0.14831 51.4876)      13.102838     2.646376  51.487600   
...                         ...            ...          ...        ...   
1875   POINT (0.27114 51.35039)       0.000000     0.000000  51.350388   
1876   POINT (0.27158 51.35937)       0.000000     0.000000  51.359372   
201   POINT (-0.43308 51.33495)       0.000000     0.000000  51.334947   
202   POINT (-0.43277 51.34394)       0.000000     0.000000  51.343936   
1      POINT (-0.5057 51.30892)       0.000000     0.000000  51.308919   

           lon  
254  -0.412511  
863  -0.145405  
465  -0.328057  
1250  0.044745  
1512  0.148308  
...      

Top 10 Coordinates

In [9]:
top10 = grid_latlon.sort_values(by='gravity_log', ascending=False).head(10)
top10[['lat','lon','gravity_log']]

,lat,lon,gravity_log
254,51.514517,-0.412511,4.610061
863,51.348731,-0.145405,3.431300
465,51.459390,-0.328057,2.869004
1250,51.426502,0.044745,2.852721
1512,51.487600,0.148308,2.646376
1254,51.462445,0.046351,2.609870
959,51.536760,-0.094514,2.390118
1225,51.543568,0.035567,2.367130
1224,51.525596,0.034765,2.364059
1227,51.588496,0.037575,2.339030
